In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# number time series per benchmark
NB_SERIES  = 250
# Percentage train
TRAIN_SIZE = 0.20

In [3]:
import warnings  # To hide the warnings of pandas group by 
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [4]:
def get_g_max(n_classes):
    return int(np.floor((n_classes+1)/3))

In [5]:
import scipy

def get_r(X, y, l_min, l_max):
    r_max = -1
    for class_label in np.unique(y):
        
        # Find the instances of the current class
        class_instances_indexes = np.where(y == class_label)[0]
        
        # Extract the instances, and resample if necessary
        if isinstance(X, list):
            length = int(np.round((l_min + l_max) / 2))
            class_instances = np.array([
                scipy.signal.resample(X[i], length, axis=1) for i in class_instances_indexes
            ])
        else: 
            class_instances = X[np.where(y == class_label)[0]]        
        
        # Compute the distance matrix
        N, D, T = class_instances.shape
        distance_matrix = scipy.spatial.distance.squareform(scipy.spatial.distance.pdist(class_instances.reshape(N, D*T), metric='euclidean'))
        
        # Find the R for this class, and update r_max
        r = np.min(np.max(distance_matrix, axis=0))
        if r > r_max:
            r_max = r
    
    return r_max

In [6]:
# We are only interested in the multivariate time series
MULTIVARIATE_DATASETS = [
    "ArticularyWordRecognition", 
    "ERing", 
    "Cricket", 
    "UWaveGestureLibrary", 
    "PenDigits", 
    "NATOPS", 
    "CharacterTrajectories", 
    "SpokenArabicDigits", 
    "JapaneseVowels"
]
UNIVARIATE_DATASETS = [
    "ECG5000",
    "Fungi",
    "Mallat",
    "Plane",
    "Symbols"
]

In [7]:
# Format the ground truth
def format_ground_truth(ts, gt):
    motifs = []
    for label, intervals in gt.items():
        mask = np.ones(shape=ts.shape[1], dtype=bool)
        location = np.empty(shape=(len(intervals), 2, ts.shape[1]), dtype=int)
        for i, (s, e) in enumerate(intervals):
            location[i, 0, :] = s
            location[i, 1, :] = e
        motifs.append((mask, location))
    return motifs

In [8]:
import tsmd_evaluation.benchmark_generation as benchmark_generation

columns = {'ds_name': str, 'c': int, 'l_min': int, 'l_max': int, 'D': int, 'g_max' : int, 'r': float, 'n_avg_train': int, 'n_avg_test': int, 'n_avg': int}
metadata = pd.DataFrame(columns, index=[])

from aeon.datasets import load_classification

path_to_benchmark = os.path.join(".")
if not os.path.exists(path_to_benchmark):
    os.mkdir(path_to_benchmark)


def znormalize(ts):
    ts = (ts - np.mean(ts, axis=None)) / np.std(ts, axis=None)
    return ts

for ds_name in MULTIVARIATE_DATASETS + UNIVARIATE_DATASETS:
        
    np.random.seed(0)    
    print(ds_name)
    # X, y
    X_train, y_train = load_classification(name=ds_name, split='train', load_equal_length=False)
    X_test, y_test = load_classification(name=ds_name, split='test', load_equal_length=False)

    df_train = benchmark_generation.convert_X_y_to_df(X_train, y_train)
    df_test = benchmark_generation.convert_X_y_to_df(X_test, y_test)

    # Combine, z-normalize, and resplit
    df = pd.concat((df_train, df_test)).reset_index(drop=True)
    df['ts'] = df['ts'].apply(znormalize)
    df_train = df.groupby('label', group_keys=False).apply(lambda x: x.sample(frac=TRAIN_SIZE)).sample(frac=1.0).reset_index(drop=True)
    df_test  = df.drop(df_train.index).sample(frac=1.0).reset_index(drop=True)
        
    # Generate tsmd benchmark
    classes = df['label'].unique()
    n_classes  = len(classes)
    g_max = get_g_max(n_classes)
    
    nb_train = int(TRAIN_SIZE * NB_SERIES) 
    nb_test  = NB_SERIES - nb_train

    benchmark_train = benchmark_generation.generate_tsmd_benchmark_dataset(df_train, nb_train, g_min=1, g_max=g_max)
    benchmark_test  = benchmark_generation.generate_tsmd_benchmark_dataset(df_test,  nb_test,  g_min=1, g_max=g_max)
    
    benchmark_train['gt'] = benchmark_train.apply(lambda r: format_ground_truth(r['ts'], r['gt']), axis='columns')
    benchmark_test['gt'] = benchmark_test.apply(lambda r: format_ground_truth(r['ts'], r['gt']), axis='columns')
    
    # Store the benchmark
    path_to_benchmark_dataset = os.path.join(path_to_benchmark, ds_name.lower())
    if not os.path.exists(path_to_benchmark_dataset):
        os.mkdir(path_to_benchmark_dataset) 

    benchmark_train.to_pickle(os.path.join(path_to_benchmark_dataset, 'validation.pkl'))
    benchmark_test.to_pickle(os.path.join(path_to_benchmark_dataset, 'test.pkl'))
        
    # Store metadata about the instances in the validation set
    d = df_train['ts'].iloc[0].shape[1]
    
    lengths = df_train['length'].to_numpy()
    l_min, l_max = np.min(lengths), np.max(lengths)
    
    r = get_r(X_train, y_train, l_min, l_max)
    
    n_avg_train = int(benchmark_train['ts'].apply(lambda x: x.shape[0]).mean())
    n_avg_test = int(benchmark_test['ts'].apply(lambda x: x.shape[0]).mean())
    n_avg = int(pd.concat([benchmark_test, benchmark_train])['ts'].apply(lambda x: x.shape[0]).mean())
    
    new_row = {'ds_name': ds_name.lower(), 'c': n_classes, 'D': d , 'l_min': l_min, 'l_max': l_max, 'g_max' : int(np.floor((n_classes+1) / 3.0)), 'r': r, 'n_avg_train': n_avg_train, 'n_avg_test': n_avg_test, 'n_avg': n_avg}
    metadata.loc[len(metadata)] = new_row

ArticularyWordRecognition
ERing
Cricket
UWaveGestureLibrary
PenDigits
NATOPS
CharacterTrajectories
SpokenArabicDigits
JapaneseVowels
ECG5000
Fungi
Mallat
Plane
Symbols


In [9]:
metadata = metadata.set_index('ds_name')
metadata.to_csv(os.path.join(path_to_benchmark, 'metadata.csv'))
metadata

,c,l_min,l_max,D,g_max,r,n_avg_train,n_avg_test,n_avg
ds_name,,,,,,,,,
articularywordrecognition,25,144,144,9,8,44.623826,4285,5790,5489
ering,6,65,65,4,2,15.939726,657,655,656
cricket,12,1197,1197,6,4,115.477160,14603,22443,20875
uwavegesturelibrary,8,315,315,3,3,48.541768,4132,4063,4077
pendigits,10,8,8,2,3,241.766416,138,135,135
natops,6,51,51,24,2,33.100815,510,507,508
charactertrajectories,20,63,180,3,7,21.916159,3987,4013,4008
spokenarabicdigits,10,4,87,13,3,51.939169,674,678,677
japanesevowels,9,7,24,12,3,4.143370,227,234,232
